# nbstripout — clean notebooks before commit
Set up git so it ignores notebook **outputs** and **run counts** (`execution_count`), and only tracks your source code. Written for AMUSE environments, but works with any Python env.

## Why?
By default git tracks the whole `.ipynb` file — including every cell's output and its run counter. Every time you re-run a notebook, `git diff` becomes a wall of noise. `nbstripout` is a git filter that strips `outputs` and `execution_count` on the way **in** (at commit time) and leaves your local file untouched on the way **out** (at checkout).

## Prerequisites
- A git repository — run this notebook from its **root**.
- A Python environment with AMUSE installed, **activated** as the notebook kernel.
- The helper script `setup_nbstripout.sh` in the same folder (for Option 1).

In [ ]:
import os
# If this notebook is not already in the repo root, cd there (example):
# os.chdir('/path/to/your/repo')
print('Working directory:', os.getcwd())


## Option 1 — one command (recommended)
Run the helper script. It installs nbstripout, configures the git filter, and re-stages all tracked notebooks:

In [ ]:
!bash setup_nbstripout.sh


## Option 2 — step by step
The same thing done manually, so you can see each piece:

In [ ]:
!python -m pip install --upgrade nbstripout


In [ ]:
# Configures filter.nbstripout.* in .git/config and adds the
# *.ipynb / *.zpln lines to .gitattributes if they are missing.
!python -m nbstripout --install


In [ ]:
# Re-stage tracked notebooks through the clean filter, so outputs and
# run counts that were already committed get stripped from the index.
!git add --renormalize .


In [ ]:
!git status


## What each piece does
- **`filter.nbstripout.clean`** — runs on `git add`/`git diff`/`git status`; strips outputs + `execution_count`.
- **`filter.nbstripout.smudge = cat`** — runs on `git checkout`; does nothing, so your local outputs are never overwritten.
- **`filter.nbstripout.required = true`** — if nbstripout is missing, git **errors** instead of silently committing outputs.
- **`diff.ipynb.textconv`** — renders readable text diffs for notebooks instead of raw JSON.
- **`.gitattributes`** — the `*.ipynb filter=nbstripout` line tells git which files to run through the filter.
- **`git add --renormalize`** — applies the clean filter to already-tracked files (one-time migration).

## Verify it's working

In [ ]:
!git config --get-regexp 'filter\.nbstripout|diff\.ipynb'
!cat .gitattributes


Then the real test: run any cell in a notebook to produce fresh output, and check that
`git status` no longer shows that file as modified.

## Ignoring AMUSE data files (`binev.data`, `SeBa.data`, …)
`.gitignore` only affects files git doesn't already track. If a data file was committed before, **untrack** it first (it stays on disk), then add it to `.gitignore`:

In [ ]:
# Replace with your own generated-data path(s).
!git rm --cached white_dwarf_project/notebooks/binev.data
!printf '%s\n' 'white_dwarf_project/notebooks/binev.data' >> .gitignore
!git check-ignore white_dwarf_project/notebooks/binev.data


## Day-to-day behaviour
- **You:** keep running notebooks normally — outputs stay on disk.
- **git:** sees the *stripped* notebook, so only source edits show in `status`/`diff`.
- **Commit:** `git commit` stores the cleaned version automatically; no extra step.

## On a fresh clone / new machine
The git filter config lives in `.git/config` (repo-local) and is **not** pushed. After cloning the repo, run Option 1 once (`bash setup_nbstripout.sh`, or just `python -m nbstripout --install`).

## Undo
To remove the filter and attributes again:  `python -m nbstripout --uninstall`.